In [ ]:
import pandas as pd
from collections import Counter
import os
from urlextract import URLExtract 
import re
import numpy as np
from datetime import datetime
os.getcwd()

Generic functions

In [ ]:
def has_url(text):
    return URLExtract().has_urls(str(text))
def CounterProc(x):
    c = Counter(x)
    return [{k:round(c[k]/sum([c[k] for k in c]),2)} for k in c]
def CounterProcOrdered(x):
    c = Counter(x).most_common()
    df = pd.DataFrame(c, columns = ['var','n'])
    df['proc'] = round(df['n'] / sum(df['n']),2)
    return df
def remove_spaces(txt):
    return(" ".join(str(txt).split()))


## Define classes for Subreddit, Submission, Comment

In [ ]:
class reddit_subr(pd.DataFrame):
    
    @property
    def _constructor(self):
        return reddit_subr
    
    def selected_subreddits(self):
        return self.query('keep')
        
    def selected_columns(self):
        return self[['subreddit','category','subscribers_K','total7d']]
    
    def rename_subreddits(self):
        return self.rename(columns = {'subscribers_K':'sr_subscribers', 'category':'sr_category','total7d':'sr_ncovid'})

In [ ]:
class reddit_subm(pd.DataFrame):
   
    @property
    def _constructor(self):
            return reddit_subm
    
    def new_names(self):
        return(self.rename(columns = {'id':'subm_id',
                             'created_utc':'subm_created',
                             'full_link':'subm_link',
                             'author':'subm_author',
                             'score':'subm_score',
                             'removed_by_category':'subm_removed',
                             'is_self':'subm_is_self',
                             'selftext':'subm_selftext',
                             'num_comments':'subm_ncomm',
                             'title':'subm_title',
                             'url':'subm_url',
                             'domain':'subm_domain',
                             'link_flair_text':'subm_flair'}))
    def new_order(self):
        return(self[['subreddit','subm_id','subm_title','subm_created',\
                  'subm_url','subm_domain','subm_author',\
                  'subm_score','subm_ncomm','subm_flair','subm_link']])

In [ ]:
class reddit_comm(pd.DataFrame):
   
    @property
    def _constructor(self):
            return reddit_comm

    def new_names(self):
        return self.rename(columns = {'id':'comm_id',
                                     'created':'comm_created',
                                     'link_id':'subm_id',
                                     'parent_id':'comm_parent_id',
                                     #'body':'comm_body',
                                     'author':'comm_author',
                                     'score':'comm_score',
                                     'collapsed':'comm_collapsed'})
    
    def define_users(self):
        return self.assign(comm_author_class = lambda df: df['distinguished'].fillna('user')).\
                    assign(comm_body = lambda df: df['body'].astype(str))
        
    def new_order(self):
        return self[['subreddit','subm_id','comm_id',\
                      'comm_body', 'comm_created','comm_parent_id','comm_author',\
                      'comm_score','comm_collapsed','comm_author_class']]

## Read Subreddits, Submissions, Comments

For now, just reading April dataset

In [ ]:
SR = pd.read_csv('../../input/subr_classification.csv')
SS = pd.read_csv('../../output/SUBM_2020_04.csv')
CC = pd.read_csv('../../output/COMM_2020_04.csv', lineterminator='\n')

### Setting conditions for keeping S & C

In [ ]:
covid_regex = '|'.join(pd.read_csv('../../input/keywords_covid.txt',header=None).loc[:,0].\
                  str.replace("\\(|\\)",r"").str.replace(r"\*",r""))#r"\\w*"))
covid_regex

In [ ]:
condition_keep_subm = \
"not is_self and \
 not is_video and \
 not post_hint.fillna('dkna').str.contains('image|video|self') and \
 not domain.fillna('dkna').str.contains('reddit|redd.it|imgur|gfycat|google|twitter|youtu|^self\\.|^i\\.redd') and \
 num_comments>0"
condition_has_covid_regex = \
'(subreddit + title + full_link + selftext.fillna("")).str.contains("'+covid_regex+'")'

## Unification

### Define functions

In [ ]:
def unite_subm_subr(SS,SR):
    return(\
        ### Read Submissions
        reddit_subm(SS).\
             ### Clean Submissions
             query(condition_keep_subm).\
                          query(condition_has_covid_regex).\
                          new_names().new_order().\
                          sort_values(['subreddit','subm_id','subm_created']).\
                          set_index(['subreddit','subm_id']).\
        ### Join Subreddits
        join(reddit_subr(SR).\
             ### Clean Subreddits
             selected_subreddits().\
             selected_columns().\
             rename_subreddits().\
             sort_values(['subreddit']).\
             set_index('subreddit'),
            how = 'left'))

In [ ]:
def unite_all(CC,SS,SR):
    return(\
        ### Read Comments
        reddit_comm(CC).\
            ### Clean Comments
            new_names().define_users().new_order().\
            sort_values(['subreddit','subm_id','comm_created']).\
            set_index(['subreddit','subm_id','comm_id']).\
        ### Join Submissions
        join(reddit_subm(SS).\
             ### Clean Submissions
             query(condition_keep_subm).\
                          query(condition_has_covid_regex).\
                          new_names().new_order().\
                          sort_values(['subreddit','subm_id','subm_created']).\
                          set_index(['subreddit','subm_id']),
            how = 'inner').\
        ### Join Subreddits
        join(reddit_subr(SR).\
             ### Clean Subreddits
             selected_subreddits().\
             selected_columns().\
             rename_subreddits().\
             sort_values(['subreddit']).\
             set_index('subreddit'),
            how = 'left'))

### Unify

#### Submissions with Subreddits

In [ ]:
SBSR = unite_subm_subr(SS,SR)
SBSR.head()

#### Comments with Submissions with Subreddits

In [ ]:
COMSBSR = unite_all(CC,SS,SR)
COMSBSR.head()

### Do not re-run these

In [ ]:
%%timeit -r 3 -n 3 
COMSBSR = unite_all(CC,SS,SR)

In [ ]:
SBSR.to_csv('../../output/SUBM_SUBR_'+str(datetime.now().date())+'.csv')

In [ ]:
COMSBSR.to_csv('../../output/UNITED_'+str(datetime.now().date())+'.csv')

### Importing United datasets

We can import united dataset so we do not need to run again uniting syntax.

In [ ]:
SBSR = pd.read_csv('../../output/SUBM_SUBR_2020-04-23.csv').set_index(['subreddit','subm_id'])
COMSBSR = pd.read_csv('../../output/UNITED_2020-04-23.csv').set_index(['subreddit','subm_id','comm_id'])
COMSBSR.shape

In [ ]:
COMSBSR.head()

#### Codebook for United dataset

In [ ]:
pd.read_csv('../../input/united_codebook.csv')

### Analysis examples

Percent of submissions with score >1, by subreddit category

In [ ]:
#SSQ.groupby('sr_category').agg({'subm_score': lambda x: round(np.exp(np.log(x+1).mean()),2)})
SBSR.groupby('sr_category').agg({'subm_score': lambda x: round((x>1).mean(),2)})

Extract shared domains with higher percentage of removals

In [ ]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(COMSBSR.assign(comm_del = lambda df: df['comm_body'] == '[removed]').\
                    groupby(['subreddit','subm_domain']).\
                    agg(count = ('comm_del', 'count'),
                        perc_removed = ('comm_del','mean')).\
                    reset_index().\
                    query('count>100').\
                    sort_values(['subreddit','perc_removed'], ascending = [True,False]).\
                    groupby(['subreddit']).head(3).
                    set_index(['subreddit','subm_domain']))

## Fake News Regex

In [ ]:
from IPython.core.display import display, HTML

def highlight_re(text, regex):
    htmld = re.sub(r'('+regex+')', '<mark>' + r'\1' + '</mark>', text.lower())
    display(HTML(htmld))
    
def print_submission(i, index, row):
    print('\n\n\nr/%s \n--------------------' % (index[0]))
    print('Published: %s' % row.subm_created[:20])
    print('%d | SUBM: %10s | COMM: %10s\n--------------------' % (i, index[1], index[2]))
    highlight_re('<b>'+row.subm_title+'</b>', covid_regex)
    print('URL: %s\nSubmission score: %d' % (row.subm_url, row.subm_score))

def print_comment(i, index, row, flag_no, limit = 1000):
    print('\nFLAG #%d (score = %d)' % (flag_no, row.comm_score))
    highlight_re(row.comm_body[:limit], fake_regex)    

In [ ]:
fake_regex = '(fake |false).{0,15}(new\\w|info\\w|hood\\w)'+'|'+\
             '(dis|mis|mal)(info\\w|leading)'+'|'+\
             '\\bbs\\b|\\bbullshit\\b'+\
             '(not ).{0,5}(true\\b|verif\\w|real\\b)'+'|'+\
             '(unreli\\w+|unverifi\\w+|untrustw\\w+|fake\\w+)( source)'
fake_regex_no = '((no|not) .{0,10}('+fake_regex+'))'+'|(users often report submissions from this site)'

Extract comments with fake news regex

In [ ]:
COMM_WITH_FLAGS = COMSBSR.query('comm_body.str.contains("'+fake_regex+'") and not comm_body.str.contains("'+fake_regex_no+'")')
COMM_WITH_FLAGS.shape

In [ ]:
print('Percent of comments containing fake regex: %.2f' % (100*COMM_WITH_FLAGS.shape[0]/COMSBSR.shape[0])+'%')

#### Select latest 1000 comments with COVID regex in submission and FAKE regex in comment

In [ ]:
COMM_WITH_FLAGS_latest1000 = COMM_WITH_FLAGS.sort_values('subm_created',ascending=False)[:1000]

#### Print selection

In [ ]:
current_submission_id = 'none'
for i,(index,row) in enumerate(COMM_WITH_FLAGS_latest1000.iterrows()):
    if current_submission_id==index[1]:
        flag_no += 1
    else:
        print_submission(i,index,row)
        flag_no = 1
    current_submission_id=index[1]
    print_comment(i,index,row,flag_no)